In [1]:

from pathlib import Path
import csv
import sys

import nibabel as nib
import numpy as np

# ==================================================
# 1. PROJECT CONFIGURATION
# ==================================================

ROOT = Path(r"D:\PS07_CT2MRI")

DATASET_DIR = ROOT / "dataset" / "volumes"
TRAIN_CSV = ROOT / "train_patients.csv"

OUTPUT_DIR = ROOT / "outputs" / "evidence_map"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_FILE = OUTPUT_DIR / "patch_index_train.npz"

# ==================================================
# 2. PATCH SETTINGS
# ==================================================

PATCH_SIZE = 32
PATCH_STRIDE = 32

MIN_BODY_FRACTION = 0.50

# TEST MODE: process only one training patient
# Change to None only after the test passes.
MAX_PATIENTS = None

# ==================================================
# 3. IMPORT OFFICIAL NORMALIZATION
# ==================================================

sys.path.insert(0, str(ROOT / "dataset"))

from normalise import normalise_ct, normalise_mri

# ==================================================
# 4. LOAD TRAINING PATIENTS
# ==================================================

if not TRAIN_CSV.exists():
    raise FileNotFoundError(
        f"Training CSV not found: {TRAIN_CSV}"
    )

with open(TRAIN_CSV, newline="", encoding="utf-8") as f:

    reader = csv.DictReader(f)

    required_columns = {"id", "split"}

    if not required_columns.issubset(reader.fieldnames or []):
        raise ValueError(
            f"Expected columns {required_columns}, "
            f"found {reader.fieldnames}"
        )

    train_ids = [
        row["id"].strip()
        for row in reader
        if row["split"].strip().lower() == "train"
    ]

if len(train_ids) != 115:
    raise ValueError(
        f"Expected 115 training patients, found {len(train_ids)}"
    )

if len(set(train_ids)) != len(train_ids):
    raise ValueError("Duplicate patient IDs found in training CSV.")

if MAX_PATIENTS is not None:
    selected_ids = train_ids[:MAX_PATIENTS]
else:
    selected_ids = train_ids

print("Total training patients:", len(train_ids))
print("Patients selected:", selected_ids)

# ==================================================
# 5. PATCH STORAGE
# ==================================================

ct_patches = []
mr_patches = []
metadata = []

total_skipped_body = 0
total_skipped_empty_mri = 0

# ==================================================
# 6. PROCESS PATIENTS
# ==================================================

for patient_number, patient_id in enumerate(
    selected_ids, start=1
):

    print("\n" + "=" * 50)
    print(f"Processing {patient_number}/{len(selected_ids)}")
    print("Patient:", patient_id)

    patient_dir = DATASET_DIR / patient_id

    ct_path = patient_dir / "ct.nii.gz"
    mr_path = patient_dir / "mr.nii.gz"
    mask_path = patient_dir / "mask.nii.gz"

    for path in [ct_path, mr_path, mask_path]:
        if not path.exists():
            raise FileNotFoundError(
                f"Missing file: {path}"
            )

    # --------------------------------------------------
    # Load NIfTI volumes
    # --------------------------------------------------

    ct_img = nib.load(str(ct_path))
    mr_img = nib.load(str(mr_path))
    mask_img = nib.load(str(mask_path))

    if not (
        ct_img.shape == mr_img.shape == mask_img.shape
    ):
        raise ValueError(
            f"Shape mismatch for {patient_id}"
        )

    if not (
        np.allclose(ct_img.affine, mr_img.affine)
        and np.allclose(ct_img.affine, mask_img.affine)
    ):
        raise ValueError(
            f"Affine mismatch for {patient_id}"
        )

    ct = np.asarray(ct_img.dataobj, dtype=np.float32)
    mr = np.asarray(mr_img.dataobj, dtype=np.float32)
    mask = np.asarray(mask_img.dataobj)

    # --------------------------------------------------
    # Official dataset normalization
    # --------------------------------------------------

    ct = normalise_ct(ct)
    mr = normalise_mri(mr, mask)

    mask = mask > 0

    if not (
        np.isfinite(ct).all()
        and np.isfinite(mr).all()
    ):
        raise ValueError(
            f"Non-finite values found in {patient_id}"
        )

    X, Y, Z = ct.shape

    print("Volume shape:", ct.shape)
    print("Normalized CT range:", ct.min(), ct.max())
    print("Normalized MRI range:", mr.min(), mr.max())

    patient_patch_count = 0

    # --------------------------------------------------
    # Extract paired 2D patches
    # --------------------------------------------------

    for z in range(Z):

        for x in range(
            0, X - PATCH_SIZE + 1, PATCH_STRIDE
        ):

            for y in range(
                0, Y - PATCH_SIZE + 1, PATCH_STRIDE
            ):

                ct_patch = ct[
                    x:x + PATCH_SIZE,
                    y:y + PATCH_SIZE,
                    z
                ]

                mr_patch = mr[
                    x:x + PATCH_SIZE,
                    y:y + PATCH_SIZE,
                    z
                ]

                mask_patch = mask[
                    x:x + PATCH_SIZE,
                    y:y + PATCH_SIZE,
                    z
                ]

                # Skip patches with insufficient body area
                if mask_patch.mean() < MIN_BODY_FRACTION:
                    total_skipped_body += 1
                    continue

                # Skip completely empty MRI patches
                if not np.any(mr_patch > 0):
                    total_skipped_empty_mri += 1
                    continue

                # Store corresponding CT and MRI patches
                ct_patches.append(ct_patch.copy())
                mr_patches.append(mr_patch.copy())

                metadata.append(
                    [patient_id, z, x, y]
                )

                patient_patch_count += 1

    print("Valid patches:", patient_patch_count)

# ==================================================
# 7. CONVERT TO ARRAYS
# ==================================================

if len(ct_patches) == 0:
    raise RuntimeError(
        "No valid patches extracted."
    )

ct_patches = np.stack(
    ct_patches
).astype(np.float32)

mr_patches = np.stack(
    mr_patches
).astype(np.float32)

metadata = np.asarray(
    metadata,
    dtype=str
)

# ==================================================
# 8. FINAL QUALITY CHECKS
# ==================================================

assert ct_patches.shape == mr_patches.shape

assert ct_patches.ndim == 3
assert ct_patches.shape[1:] == (32, 32)

assert len(metadata) == len(ct_patches)

assert np.isfinite(ct_patches).all()
assert np.isfinite(mr_patches).all()

assert np.all(ct_patches >= 0)
assert np.all(ct_patches <= 1)

assert np.all(mr_patches >= 0)
assert np.all(mr_patches <= 1)

assert np.all(mr_patches.max(axis=(1, 2)) > 0)

assert len(np.unique(metadata, axis=0)) == len(metadata)

# ==================================================
# 9. SAVE PATCH INDEX
# ==================================================

np.savez_compressed(
    OUTPUT_FILE,
    ct_patches=ct_patches,
    mr_patches=mr_patches,
    metadata=metadata
)

# ==================================================
# 10. FINAL REPORT
# ==================================================

print("\n" + "=" * 55)
print("PATCH INDEX GENERATION COMPLETED")
print("=" * 55)

print("Patients processed:", len(selected_ids))

print("CT patch shape:", ct_patches.shape)
print("MRI patch shape:", mr_patches.shape)
print("Metadata shape:", metadata.shape)

print("CT range:",
      float(ct_patches.min()),
      float(ct_patches.max()))

print("MRI range:",
      float(mr_patches.min()),
      float(mr_patches.max()))

print("Skipped body patches:", total_skipped_body)
print("Skipped empty MRI patches:", total_skipped_empty_mri)

print("CT finite:", np.isfinite(ct_patches).all())
print("MRI finite:", np.isfinite(mr_patches).all())

print("Unique metadata:",
      len(np.unique(metadata, axis=0)))

print("\nSaved to:", OUTPUT_FILE)


Total training patients: 115
Patients selected: ['P1046', 'P1068', 'P1127', 'P1316', 'P1375', 'P1389', 'P1815', 'P1965', 'P2017', 'P2052', 'P2161', 'P2341', 'P2349', 'P2374', 'P2420', 'P2622', 'P2710', 'P2728', 'P2783', 'P2908', 'P2919', 'P2988', 'P3015', 'P3198', 'P3205', 'P3254', 'P3364', 'P3462', 'P3517', 'P3521', 'P3679', 'P3884', 'P4020', 'P4021', 'P4204', 'P4223', 'P4369', 'P4390', 'P4408', 'P4460', 'P4506', 'P4582', 'P4603', 'P4697', 'P4919', 'P4929', 'P4938', 'P4940', 'P4944', 'P4985', 'P5137', 'P5220', 'P5231', 'P5331', 'P5415', 'P5456', 'P5465', 'P5510', 'P5540', 'P5565', 'P5899', 'P5963', 'P6054', 'P6142', 'P6156', 'P6410', 'P6430', 'P6446', 'P6494', 'P6511', 'P6514', 'P6586', 'P6699', 'P6706', 'P6769', 'P6906', 'P6938', 'P7020', 'P7036', 'P7051', 'P7119', 'P7152', 'P7202', 'P7306', 'P7411', 'P7610', 'P7739', 'P7846', 'P8019', 'P8151', 'P8174', 'P8256', 'P8309', 'P8385', 'P8398', 'P8490', 'P8546', 'P8582', 'P8586', 'P8601', 'P8715', 'P8838', 'P8840', 'P8917', 'P9524', 'P9609